<!-- nav -->
[← 1 · Programs as data](01_Programs_As_Data.ipynb) · [Home](../../README.md) · [3 · Which standard? →](03_Which_Standard.ipynb)

# 2 · Changing code safely

**The problem.** A team replaces `print` with a logger across a code base. A regular expression finds `print(` in
strings, comments and names like `blueprint(`, and misses `print (`. Editing code as text is guessing at its structure.

On a tree, a call to `print` is a `Call` whose function is the `Name` `print`: there's nothing to guess. This case
study finds, changes, replaces and copies syntax nodes.

In [1]:
import { Parents, Transformer, copy, walk, type SyntaxNode } from "@mbse/programs/Framework/Syntax";
import { Python314, Syntax as P } from "@mbse/programs/Python";

const Py = P.LANGUAGE.Builders;
const source = `def report(total):
    print("total:", total)
    blueprint("print(total)")
    return total


print(report(42))
`;
const module = Python314.parse(source) as P.Module;

## Step 1: finding

`walk` visits every syntax node of a tree, parents before children. A call to `print` is easy to say exactly:

In [2]:
function callsTo(name: string, tree: SyntaxNode): P.Call[] {
  return [...walk(tree)].filter((node): node is P.Call =>
    node instanceof P.Call && node.func instanceof P.Name && node.func.id.spelling === name);
}

console.log(callsTo("print", module).length, "calls to print");
console.log(callsTo("print", module).map((call) => Python314.print(call.args[0]).trim()));

2 calls to print


[ '"total:"', "report(42)" ]


Two, and not `blueprint`, nor the `print(total)` inside the string.

## Step 2: changing in place

Syntax nodes are plain objects, so the simplest change is an assignment. Each `print` becomes `log.info`, by replacing
the call's function:

In [3]:
for (const call of callsTo("print", module)) {
  call.func = Py.Attribute().value((b) => b.Name().id("log")).attr("info").create();
}
console.log(Python314.print(module));

def report(total):
    log.info("total:", total)
    blueprint("print(total)")
    return total


log.info(report(42))



## Step 3: a transformer, kind by kind

For changes all over a tree, a `Transformer` has a `visit_<Kind>` method per kind it changes, and returns what replaces
each syntax node. Here every `return x` gains a log line before it. A statement can be replaced by several, so the
method returns a list:

In [4]:
class LogReturns extends Transformer {
  visit_Return(node: P.Return): SyntaxNode[] {
    const logged = Py.Expr().value((b) => b.Call().func((b) => b.Attribute().value((b) => b.Name().id("log")).attr("debug"))
      .add_args(copy(node.value as P.Expression))).create();
    return [logged, node];
  }
}

new LogReturns().visit(module);
console.log(Python314.print(module));

def report(total):
    log.info("total:", total)
    blueprint("print(total)")
    log.debug(total)
    return total


log.info(report(42))



`copy(node.value)` duplicates the returned expression, because the log line and the return can't share one syntax
node: step 5 says why.

## Step 4: where a syntax node is

`Parents` records each syntax node's parent, the property holding it and its index, and its **path** from the root,
which is how errors name places in a tree. It also replaces and removes:

In [5]:
const parents = new Parents(module);
const blueprint = [...walk(module)].find((node) => node instanceof P.Name && node.id.spelling === "blueprint")!;
const statement = parents.parent(parents.parent(blueprint)!)!;
console.log(parents.path(blueprint), "in", parents.path(statement));
parents.remove(statement);
console.log(Python314.print(module));

body[0].body[1].value.func in body[0].body[1]


def report(total):
    log.info("total:", total)
    log.debug(total)
    return total


log.info(report(42))



## Step 5: one place per syntax node

A tree is a tree: each syntax node has one parent. Using a syntax node twice makes two places that would change
together, and `validate` refuses it. `copy` makes the second one:

In [6]:
const name = Py.Name().id("n").create();
const shared = Py.BinOp().left(name).op("*").right(name).create();
console.log(P.LANGUAGE.validate(shared));
const squared = Py.BinOp().left(name).op("*").right(copy(name)).create();
console.log(P.LANGUAGE.validate(squared), Python314.print(squared));

[ "right: the syntax node is also at left" ]


[] n * n


## Why it works this way

- **Structure, not spelling.** A call is a `Call`, whatever its spacing; a string is a `Constant`, whatever it contains.
  Changes made on the tree can't touch what they don't mean to.
- **Several tools, one tree.** `walk` for finding, assignment for small changes, `Transformer` for sweeping ones,
  `Parents` for where things are: each the right size for its job, all on the same plain objects.
- **One parent each.** Two places that silently share a syntax node would change together. Copying is explicit.

Renaming is harder than it looks: the next case studies cover which standard a change needs, and then what names refer
to.

<!-- nav -->
[← 1 · Programs as data](01_Programs_As_Data.ipynb) · [Home](../../README.md) · [3 · Which standard? →](03_Which_Standard.ipynb)